In [1]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025

# Load play-by-play data for the chosen season
pbp = nfl.import_pbp_data(years=[2025])

# Filter for regular season, Week 1
week1 = pbp[(pbp['week'] == 1)]

# Keep only touchdown plays
week1_tds = week1[week1['touchdown'] == 1]

# Count TDs per scorer. Prefer id+name if both available, else fall back to name only
use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week1_tds.columns]
if use_cols:
    scorers = (
        week1_tds.dropna(subset=use_cols)
        .groupby(use_cols)
        .size()
        .reset_index(name='tds')
    )
    if 'td_player_id' in use_cols:
        scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
    else:
        scorers = scorers.rename(columns={'td_player_name': 'player'})
else:
    # Fallback if td_* columns not present; derive from rusher/receiver
    rush = week1_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
    rec = week1_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
    rush.columns = ['player_id', 'player']
    rec.columns = ['player_id', 'player']
    both = pd.concat([rush, rec], ignore_index=True)
    scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

# Show results
scorers.head(50)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0030061,Z.Ertz,1
1,00-0030279,K.Allen,1
2,00-0030506,T.Kelce,1
3,00-0030564,D.Hopkins,1
4,00-0032764,D.Henry,2
5,00-0033288,G.Kittle,1
6,00-0033293,A.Jones,1
7,00-0033553,J.Conner,1
8,00-0033858,J.Smith,1
9,00-0033873,P.Mahomes,1


In [3]:
predictions = pd.read_csv('predictions.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions.head(50)



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
0,00-0032764,Derrick Henry,RB,BAL,0.577770,-230.0,-0.119199
1,00-0034844,Saquon Barkley,RB,PHI,0.570217,-160.0,-0.045167
2,00-0037248,James Cook,RB,BUF,0.561577,-130.0,-0.003640
3,00-0038597,Chase Brown,RB,CIN,0.559472,-150.0,-0.040528
4,00-0035700,Josh Jacobs,RB,GB,0.557630,-200.0,-0.109037
5,00-0037840,Kyren Williams,RB,LA,0.550300,-155.0,-0.057543
6,00-0038542,Bijan Robinson,RB,ATL,0.533729,-105.0,0.021534
7,00-0036358,CeeDee Lamb,WR,DAL,0.530089,100.0,0.030089
8,00-0033553,James Conner,RB,ARI,0.529656,-135.0,-0.044812
9,00-0036555,Chuba Hubbard,RB,CAR,0.516380,105.0,0.028575


In [4]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False])

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,tds
0,00-0032764,Derrick Henry,RB,BAL,0.577770,-230.0,-0.119199,2
1,00-0034844,Saquon Barkley,RB,PHI,0.570217,-160.0,-0.045167,1
2,00-0037248,James Cook,RB,BUF,0.561577,-130.0,-0.003640,1
3,00-0038597,Chase Brown,RB,CIN,0.559472,-150.0,-0.040528,1
4,00-0035700,Josh Jacobs,RB,GB,0.557630,-200.0,-0.109037,1
5,00-0037840,Kyren Williams,RB,LA,0.550300,-155.0,-0.057543,1
6,00-0038542,Bijan Robinson,RB,ATL,0.533729,-105.0,0.021534,1
7,00-0033553,James Conner,RB,ARI,0.529656,-135.0,-0.044812,1
8,00-0036555,Chuba Hubbard,RB,CAR,0.516380,105.0,0.028575,1
9,00-0039064,Zay Flowers,WR,BAL,0.493030,145.0,0.084867,1


In [5]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [6]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[predictions['model_edge'] > 0.10] 
ev = ev[ev['price'] < 500]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
17,00-0037740,Garrett Wilson,WR,NYJ,0.475642,200.0,0.142309
47,00-0035644,Noah Fant,TE,CIN,0.344927,380.0,0.136593
21,00-0039901,Keon Coleman,WR,BUF,0.459965,200.0,0.126631
49,00-0038117,Wan'Dale Robinson,WR,NYG,0.331958,370.0,0.119192


In [7]:
simulate_betting(ev, scorers)

{'bets': 4, 'hits': 3, 'hit_rate': 0.75, 'total_profit': 68.0, 'roi': 1.7}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0037740,Garrett Wilson,NYJ,WR,200.0,0.475642,0.142309,1,True,20.0,30.0
1,00-0039901,Keon Coleman,BUF,WR,200.0,0.459965,0.126631,1,True,20.0,30.0
2,00-0035644,Noah Fant,CIN,TE,380.0,0.344927,0.136593,1,True,38.0,48.0
3,00-0038117,Wan'Dale Robinson,NYG,WR,370.0,0.331958,0.119192,0,False,-10.0,0.0


In [8]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(15)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
0,00-0032764,Derrick Henry,RB,BAL,0.577770,-230.0,-0.119199
1,00-0034844,Saquon Barkley,RB,PHI,0.570217,-160.0,-0.045167
2,00-0037248,James Cook,RB,BUF,0.561577,-130.0,-0.003640
3,00-0038597,Chase Brown,RB,CIN,0.559472,-150.0,-0.040528
4,00-0035700,Josh Jacobs,RB,GB,0.557630,-200.0,-0.109037
5,00-0037840,Kyren Williams,RB,LA,0.550300,-155.0,-0.057543
6,00-0038542,Bijan Robinson,RB,ATL,0.533729,-105.0,0.021534
8,00-0033553,James Conner,RB,ARI,0.529656,-135.0,-0.044812
9,00-0036555,Chuba Hubbard,RB,CAR,0.516380,105.0,0.028575
10,00-0039139,Jahmyr Gibbs,RB,DET,0.513230,-175.0,-0.123133


In [9]:
simulate_betting(top_rb, scorers)

{'bets': 10, 'hits': 9, 'hit_rate': 0.9, 'total_profit': 53.84, 'roi': 0.538}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-230.0,0.577770,-0.119199,2,True,4.347826,14.347826
1,00-0034844,Saquon Barkley,PHI,RB,-160.0,0.570217,-0.045167,1,True,6.250000,16.250000
2,00-0037248,James Cook,BUF,RB,-130.0,0.561577,-0.003640,1,True,7.692308,17.692308
3,00-0038597,Chase Brown,CIN,RB,-150.0,0.559472,-0.040528,1,True,6.666667,16.666667
4,00-0035700,Josh Jacobs,GB,RB,-200.0,0.557630,-0.109037,1,True,5.000000,15.000000
5,00-0037840,Kyren Williams,LA,RB,-155.0,0.550300,-0.057543,1,True,6.451613,16.451613
6,00-0038542,Bijan Robinson,ATL,RB,-105.0,0.533729,0.021534,1,True,9.523810,19.523810
7,00-0033553,James Conner,ARI,RB,-135.0,0.529656,-0.044812,1,True,7.407407,17.407407
8,00-0036555,Chuba Hubbard,CAR,RB,105.0,0.516380,0.028575,1,True,10.500000,20.500000
9,00-0039139,Jahmyr Gibbs,DET,RB,-175.0,0.513230,-0.123133,0,False,-10.000000,0.000000


In [10]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
#top_wr = top_wr[top_wr['model_edge'] > 0.05]
#top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)

{'bets': 15, 'hits': 8, 'hit_rate': 0.533, 'total_profit': 55.5, 'roi': 0.37}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036358,CeeDee Lamb,DAL,WR,100.0,0.530089,0.030089,0,False,-10.0,0.0
1,00-0039064,Zay Flowers,BAL,WR,145.0,0.493030,0.084867,1,True,14.5,24.5
2,00-0039915,Ladd McConkey,LAC,WR,155.0,0.477902,0.085745,0,False,-10.0,0.0
3,00-0031381,Davante Adams,LA,WR,130.0,0.476624,0.041841,0,False,-10.0,0.0
4,00-0037740,Garrett Wilson,NYJ,WR,200.0,0.475642,0.142309,1,True,20.0,30.0
5,00-0036322,Justin Jefferson,MIN,WR,115.0,0.474494,0.009378,1,True,11.5,21.5
6,00-0039901,Keon Coleman,BUF,WR,200.0,0.459965,0.126631,1,True,20.0,30.0
7,00-0037238,Drake London,ATL,WR,175.0,0.457576,0.093939,0,False,-10.0,0.0
8,00-0035719,Deebo Samuel Sr.,WAS,WR,165.0,0.438665,0.061306,1,True,16.5,26.5
9,00-0039849,Marvin Harrison Jr.,ARI,WR,130.0,0.438236,0.003454,1,True,13.0,23.0


In [11]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 29.5, 'roi': 0.59}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0039338,Brock Bowers,LV,TE,195.0,0.381040,0.042057,0,False,-10.0,0.0
1,00-0035644,Noah Fant,CIN,TE,380.0,0.344927,0.136593,1,True,38.0,48.0
2,00-0038996,Tucker Kraft,GB,TE,215.0,0.337757,0.020296,1,True,21.5,31.5
3,00-0035229,T.J. Hockenson,MIN,TE,210.0,0.309613,-0.012967,0,False,-10.0,0.0
4,00-0038041,Jake Ferguson,DAL,TE,195.0,0.277228,-0.061755,0,False,-10.0,0.0


In [12]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 7.99, 'roi': 0.16}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034857,Josh Allen,BUF,QB,-110.0,0.454677,-0.069133,2,True,9.090909,19.090909
1,00-0036389,Jalen Hurts,PHI,QB,-145.0,0.432647,-0.159189,2,True,6.896552,16.896552
2,00-0034796,Lamar Jackson,BAL,QB,120.0,0.388714,-0.065831,1,True,12.000000,22.000000
3,00-0029263,Russell Wilson,NYG,QB,600.0,0.294230,0.151373,0,False,-10.000000,0.000000
4,00-0039910,Jayden Daniels,WAS,QB,175.0,0.265108,-0.098529,0,False,-10.000000,0.000000


In [13]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 11, 'hits': 9, 'hit_rate': 0.818, 'total_profit': 43.84, 'roi': 0.399}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-230.0,0.577770,-0.119199,2,True,4.347826,14.347826
1,00-0034844,Saquon Barkley,PHI,RB,-160.0,0.570217,-0.045167,1,True,6.250000,16.250000
2,00-0037248,James Cook,BUF,RB,-130.0,0.561577,-0.003640,1,True,7.692308,17.692308
3,00-0038597,Chase Brown,CIN,RB,-150.0,0.559472,-0.040528,1,True,6.666667,16.666667
4,00-0035700,Josh Jacobs,GB,RB,-200.0,0.557630,-0.109037,1,True,5.000000,15.000000
5,00-0037840,Kyren Williams,LA,RB,-155.0,0.550300,-0.057543,1,True,6.451613,16.451613
6,00-0038542,Bijan Robinson,ATL,RB,-105.0,0.533729,0.021534,1,True,9.523810,19.523810
7,00-0036358,CeeDee Lamb,DAL,WR,100.0,0.530089,0.030089,0,False,-10.000000,0.000000
8,00-0033553,James Conner,ARI,RB,-135.0,0.529656,-0.044812,1,True,7.407407,17.407407
9,00-0036555,Chuba Hubbard,CAR,RB,105.0,0.516380,0.028575,1,True,10.500000,20.500000


In [33]:
feature_df = pd.read_csv('feature_df.csv')
feature_df.drop(columns=['opponent_encoded', 'position_encoded'],inplace=True)

feature_df.shape


(26711, 91)

In [40]:
%pip install -q boto3
import nfl_td_lambda.data_collection as data
import predict as predict

nfl_teams_df = pd.read_csv('nfl_teams.csv')
team_map = dict(zip(nfl_teams_df['team_name'], nfl_teams_df['team_id']))
feature_df_2 = pd.read_csv('raw_nfl_data.csv')


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [41]:
### find differences in feature_df and feature_df_2
feature_df_2.equals(feature_df)
feature_df.tail()



,player_id,player_display_name,position,recent_team,season,week,carries,rushing_yards,rushing_tds,receptions,...,rushing_tds_allowed_to_TE,rushing_tds_allowed_to_WR,passing_tds_allowed_to_DB,passing_tds_allowed_to_DL,passing_tds_allowed_to_LB,passing_tds_allowed_to_OL,passing_tds_allowed_to_QB,passing_tds_allowed_to_RB,passing_tds_allowed_to_TE,passing_tds_allowed_to_WR
26706,00-0040734,TreVeyon Henderson,RB,NE,2025,1,5,27.0,0,6,...,0.000000e+00,1.604466e-07,0.000000,0.0,0.0,0.000000,0.000000e+00,0.134310,0.515282,0.945464
26707,00-0040735,Luther Burden III,WR,CHI,2025,1,0,0.0,0,1,...,2.873944e-10,1.604508e-07,0.000309,0.0,0.0,0.000000,0.000000e+00,0.172527,0.177432,0.956455
26708,00-0040736,Mason Taylor,TE,NYJ,2025,1,0,0.0,0,1,...,0.000000e+00,4.825236e-13,0.000000,0.0,0.0,0.000000,2.873889e-10,0.004733,0.656460,1.373698
26709,00-0040739,Elijah Arroyo,TE,SEA,2025,1,0,0.0,0,1,...,0.000000e+00,2.026420e-01,0.000000,0.0,0.0,0.000000,0.000000e+00,0.026068,0.411908,1.490724
26710,00-0040782,Isaiah Bond,WR,CLE,2025,1,0,0.0,0,1,...,1.034021e-16,1.949964e-04,0.000000,0.0,0.0,0.003084,0.000000e+00,0.138553,0.509517,1.123718


In [47]:
feature_df_2 = predict.transform_features(feature_df_2)

In [48]:
x = feature_df_2[feature_df_2['player_id'] == '00-0023459']
x = x[x['season'] == 2025]
x

,player_id,player_display_name,position,recent_team,season,week,carries,rushing_yards,rushing_tds,receptions,...,passing_tds_allowed_to_DB,passing_tds_allowed_to_DL,passing_tds_allowed_to_LB,passing_tds_allowed_to_OL,passing_tds_allowed_to_QB,passing_tds_allowed_to_RB,passing_tds_allowed_to_TE,passing_tds_allowed_to_WR,rush_matchup_value,pass_matchup_value
26357,00-0023459,Aaron Rodgers,QB,PIT,2025,1,1,-1.0,0,0,...,0.0,2.514078e-13,0.0,1.172968e-14,1.970136e-16,0.400051,0.777056,1.388935,0.000024,0.0


In [49]:
x = feature_df[feature_df['player_id'] == '00-0023459']
x = x[x['season'] == 2025]
x

,player_id,player_display_name,position,recent_team,season,week,carries,rushing_yards,rushing_tds,receptions,...,rushing_tds_allowed_to_TE,rushing_tds_allowed_to_WR,passing_tds_allowed_to_DB,passing_tds_allowed_to_DL,passing_tds_allowed_to_LB,passing_tds_allowed_to_OL,passing_tds_allowed_to_QB,passing_tds_allowed_to_RB,passing_tds_allowed_to_TE,passing_tds_allowed_to_WR
26357,00-0023459,Aaron Rodgers,QB,PIT,2025,1,1,-1.0,0,0,...,4.630825e-09,0.006855,0.0,1.032448e-10,0.0,5.914890e-12,1.268793e-13,0.002695,0.450422,0.97869


In [50]:
feature_df = predict.transform_features(feature_df)
x = feature_df[feature_df['player_id'] == '00-0023459']
x = x[x['season'] == 2025]
x

,player_id,player_display_name,position,recent_team,season,week,carries,rushing_yards,rushing_tds,receptions,...,rushing_tds_allowed_to_TE,rushing_tds_allowed_to_WR,passing_tds_allowed_to_DB,passing_tds_allowed_to_DL,passing_tds_allowed_to_LB,passing_tds_allowed_to_OL,passing_tds_allowed_to_QB,passing_tds_allowed_to_RB,passing_tds_allowed_to_TE,passing_tds_allowed_to_WR
26357,00-0023459,Aaron Rodgers,QB,PIT,2025,1,1,-1.0,0,0,...,3.040880e-08,0.01682,0.0,7.846606e-10,0.0,4.968509e-11,1.201136e-12,0.007743,0.382716,0.864852
